In [3]:
import numpy as np
import tensorflow as tf
import h5py
from tensorflow.keras.layers import Conv3D, MaxPool3D, Flatten, Dense, Reshape, Conv2D, Conv2DTranspose, MaxPooling2D
from tensorflow.keras.layers import Dropout, Input, BatchNormalization
from sklearn.metrics import confusion_matrix, accuracy_score
from plotly.offline import iplot, init_notebook_mode
from tensorflow.keras.losses import categorical_crossentropy
from tensorflow.keras.optimizers import Adadelta
import plotly.graph_objs as go
from matplotlib.pyplot import cm
from tensorflow.keras.models import Model
from tensorflow.keras import models
from tensorflow.keras.optimizers import Adam
from matplotlib import pyplot as plt
import math

class three_dimentions_CNN:
    """欠陥情報から最大主応力を出力するプログラム"""
    def __init__(self,works_num,lr):
        self.works_num = works_num
        self.lr = lr

    def CNN(self):
        works = []
        voids = []
        stress = []
        for i in range(1,self.works_num+1):
            a = "work"+str(i)
            works.append(a)
        print(works)
        m = 1
        for j in range(0,m):
            for i in works:
                """voidの読み込み×100"""
                voids.append(np.load(r'/home/kojima/3D_CNN/data/void_numpy//'+i+'.npy'))
        voids = np.array(voids)
        voids = np.load(r'/home/kojima/3D_CNN/data/void_multi.npy')#何倍のvoidにするか
        x_train = voids
        mm = 10
        x_train = np.transpose(x_train, (0,2,3,1))
        x_test = x_train
        self.xtest = x_test

        print(x_train.shape)

        for j in range(0,m):
            for i in works:
                """max_principal_stressの読み込み×100"""
                stress.append(np.load(r'/home/kojima/3D_CNN/data/principal_numpy//'+i+'.npy').flatten())
        y_train = np.array(stress)
        y_test = y_train
        self.y_test = y_test
        print(y_train.shape)
        xtrain = x_train
        xtest = x_test
        np.random.seed(1)
        np.random.shuffle(xtrain)
        np.random.seed(1)
        np.random.shuffle(y_train)
        xtrain_check =[]
        for i in range(2496):
            xtrain_check.append(i)
        xtrain_check = np.array(xtrain_check)

        np.random.seed(1)
        np.random.shuffle(xtrain_check)
        xtrain_val = np.sort(xtrain_check[1996:2496])

        print(xtrain_val.tolist())
        model = models.Sequential()
       

        ## convolutional layers
        model.add(Conv2D(filters=32, kernel_size=(5, 5), activation='relu',input_shape=(10*mm, 12*mm,10)))
        model.add(Conv2D(filters=64, kernel_size=(5, 5), activation='relu'))
        model.add(MaxPooling2D(pool_size = (2,2)))

        ## add max pooling to obtain the most imformatic features
        model.add(Conv2D(filters=128, kernel_size=(5, 5), activation='relu'))
        model.add(Conv2D(filters=64,kernel_size=(5,5),activation='relu'))
        model.add(MaxPooling2D(pool_size = (2,2)))
        model.add(Conv2D(filters=32, kernel_size=(4, 5), activation='relu'))
        model.add(Conv2D(filters=16,kernel_size=(4,5),activation='relu'))
        model.add(Conv2D(filters=1,kernel_size=(4,5),activation='relu'))
        model.add(Flatten())

        ## define the model with input layer and output layer
        model.compile(loss='mean_squared_error', optimizer=Adam(lr=self.lr, beta_1=0.9, beta_2=0.999, epsilon=None, decay=0.0, amsgrad=False), metrics=['acc'])
        model.fit(x=xtrain, y=y_train, batch_size=16, epochs=1500, validation_split=0.2)
        self.model = model
        model.summary()
        plt.plot(model.history.history['acc'], marker='.', label='acc')
        plt.plot(model.history.history['val_acc'], marker='.', label='val_acc')
        plt.title('model accuracy')
        plt.grid()
        plt.xlabel('epoch')
        plt.ylabel('accuracy')
        plt.legend(loc='best')
        plt.savefig(r'/home/kojima/3D_CNN/data/accuracy_all_lr.jpg')       
        plt.show()

        plt.plot(model.history.history['loss'], marker='.', label='loss')
        plt.plot(model.history.history['val_loss'], marker='.', label='val_loss')
        plt.title('model loss')
        plt.grid()
        plt.xlabel('epoch')
        plt.ylabel('loss')
        plt.legend(loc='best')
        plt.savefig(r'/home/kojima/3D_CNN/data/loss_lr_all.jpg')
        plt.show()

    def check(self):
        works = []
        voids = []
        for i in range(1,2497):
            a = "work"+str(i)
            works.append(a)
        print(works)
        m = 1
        for j in range(0,m):
            for i in works:
                """voidの読み込み×100"""
                voids.append(np.load(r'/home/kojima/3D_CNN/data/void_numpy//'+i+'.npy'))
        voids = np.array(voids)
        x_train = voids
        x_train = np.reshape(x_train, (-1,10, 10, 12))
        x_train = np.transpose(x_train,(0,2,3,1))
        x_test = x_train
        print(x_train.shape)
        pred = self.model.predict(x_test)
        np.save(r"/home/kojima/3D_CNN/data/result_all",pred)
        print(pred)
        self.model.save_weights(r'/home/kojima/3D_CNN/data/param_all.h5')
        self.model.save(r'/home/kojima/3D_CNN/data/model_all.h5')





check = three_dimentions_CNN(2496,0.00005)
check.CNN()

ModuleNotFoundError: No module named 'tensorflow'